In [0]:
# Notebook 3: Feature Engineering
# Food Delivery Analysis
# ---------------------------------------------------

from pyspark.sql.functions import (
    col, count, avg, sum, max, min, stddev,
    when, lag, datediff, to_date, round,
    dayofweek, hour, month, unix_timestamp,
    collect_list, sort_array, struct
)
from pyspark.sql.window import Window
import pyspark.sql.functions as F

# Load from Parquet
df = spark.read.parquet(
    "/Volumes/workspace/default/food_delivery_data/uae_food_delivery_750k.parquet"
)

print(f"Loaded {df.count():,} rows successfully.")

In [0]:
# User level aggregations for churn prediction using Window functions
from pyspark.sql.window import Window

user_window = Window.partitionBy("user_id")

# Find most ordered cuisine per user
from pyspark.sql.functions import row_number, desc

cuisine_counts = df.groupBy("user_id", "cuisine") \
    .agg(count("order_id").alias("cuisine_order_count"))

cuisine_window = Window.partitionBy("user_id").orderBy(desc("cuisine_order_count"))

top_cuisine = cuisine_counts \
    .withColumn("rank", row_number().over(cuisine_window)) \
    .filter(col("rank") == 1) \
    .select("user_id", col("cuisine").alias("top_cuisine"))

# Build user features
user_features = df.withColumn("total_orders", count("order_id").over(user_window)) \
    .withColumn("avg_order_value", round(avg("total_price_aed").over(user_window), 2)) \
    .withColumn("total_spend", round(sum("total_price_aed").over(user_window), 2)) \
    .withColumn("total_cancellations", count(when(col("order_status") == "Cancelled", 1)).over(user_window)) \
    .withColumn("cancellation_rate_pct", round(count(when(col("order_status") == "Cancelled", 1)).over(user_window) / count("order_id").over(user_window) * 100, 2)) \
    .withColumn("weekend_orders", count(when(col("is_weekend") == 1, 1)).over(user_window)) \
    .withColumn("ramadan_orders", count(when(col("is_ramadan_period") == 1, 1)).over(user_window)) \
    .withColumn("avg_delivery_duration", round(avg("delivery_duration_mins").over(user_window), 2)) \
    .withColumn("avg_risk_score", round(avg("order_quality_risk_score").over(user_window), 3)) \
    .dropDuplicates(["user_id"]) \
    .select(
        "user_id", "total_orders", "avg_order_value", "total_spend",
        "total_cancellations", "cancellation_rate_pct", "weekend_orders",
        "ramadan_orders", "avg_delivery_duration", "avg_risk_score",
        "user_subscription", "city", "payment_method", "churn_risk"
    )

# Join top cuisine onto user features
user_features = user_features.join(top_cuisine, on="user_id", how="left")

print(f"Total users: {user_features.count():,}")
print(f"Total features per user: {len(user_features.columns)}")
user_features.display()

In [0]:
# Restaurant level aggregations for health scoring
from pyspark.sql.window import Window
from pyspark.sql.functions import round, avg, count, when, col, countDistinct, row_number, desc

restaurant_window = Window.partitionBy("restaurant_id")

# Most ordered item per restaurant
item_counts = df.groupBy("restaurant_id", "item_name") \
    .agg(count("order_id").alias("item_order_count"))

item_window = Window.partitionBy("restaurant_id").orderBy(desc("item_order_count"))

top_item = item_counts \
    .withColumn("rank", row_number().over(item_window)) \
    .filter(col("rank") == 1) \
    .select("restaurant_id", col("item_name").alias("top_ordered_item"))

# Compute distinct items per restaurant separately (countDistinct not supported in window functions)
distinct_items = df.groupBy("restaurant_id") \
    .agg(countDistinct("item_name").alias("distinct_items_served"))

# Build restaurant features
restaurant_features = df.withColumn("total_orders", count("order_id").over(restaurant_window)) \
    .withColumn("total_cancellations", count(when(col("order_status") == "Cancelled", 1)).over(restaurant_window)) \
    .withColumn("cancellation_rate_pct", round(count(when(col("order_status") == "Cancelled", 1)).over(restaurant_window) / count("order_id").over(restaurant_window) * 100, 2)) \
    .withColumn("avg_delivery_duration", round(avg("delivery_duration_mins").over(restaurant_window), 2)) \
    .withColumn("avg_order_value", round(avg("total_price_aed").over(restaurant_window), 2)) \
    .withColumn("avg_quality_risk", round(avg("order_quality_risk_score").over(restaurant_window), 3)) \
    .dropDuplicates(["restaurant_id"]) \
    .select(
        "restaurant_id", "restaurant_name", "cuisine", "city", "area",
        "total_orders", "total_cancellations", "cancellation_rate_pct",
        "avg_delivery_duration", "avg_order_value", "avg_quality_risk",
        "restaurant_health_score"
    )

# Join distinct items count
restaurant_features = restaurant_features.join(distinct_items, on="restaurant_id", how="left")

# Join top ordered item
restaurant_features = restaurant_features.join(top_item, on="restaurant_id", how="left")

print(f"Total restaurants: {restaurant_features.count():,}")
print(f"Total features per restaurant: {len(restaurant_features.columns)}")
restaurant_features.display()

In [0]:
# Order level features for order quality risk scoring
order_features = df.select(
    "order_id",
    "user_id",
    "restaurant_id",
    "cuisine",
    "city",
    "area",
    "order_hour",
    "is_weekend",
    "is_ramadan_period",
    "delivery_distance_km",
    "traffic_level",
    "driver_vehicle",
    "driver_availability",
    "delivery_duration_mins",
    "total_price_aed",
    "quantity",
    "payment_method",
    "restaurant_health_score",
    "order_status",
    "order_quality_risk_score"
)

print(f"Total orders: {order_features.count():,}")
print(f"Total features per order: {len(order_features.columns)}")
order_features.display()

In [0]:
# Save all three feature tables as Parquet files
user_features.write.mode("overwrite").parquet(
    "/Volumes/workspace/default/food_delivery_data/user_features.parquet"
)

restaurant_features.write.mode("overwrite").parquet(
    "/Volumes/workspace/default/food_delivery_data/restaurant_features.parquet"
)

order_features.write.mode("overwrite").parquet(
    "/Volumes/workspace/default/food_delivery_data/order_features.parquet"
)

print("All three feature tables saved successfully.")
print("user_features: 25,000 rows")
print("restaurant_features: 2,000 rows")
print("order_features: 750,000 rows")